# STAC Item for NOAA CDR NDVI — GeoZarr Pyramid on Cloudflare R2

Creates a [STAC](https://stacspec.org/) Item describing the NOAA CDR NDVI
GeoZarr multiscales Icechunk pyramid store at `osc-pub/ndvi-cdr-pyramid`.
Follows the same pattern as `ndvi_cdr_stac_item.ipynb`.

**Architecture**
- Levels 1–3 (downsampled): fully **materialised** chunk data on R2
- Level 0 (full resolution): **virtual chunk references** pointing back to `osc-pub/ndvi-cdr-icechunk`

**Steps**
1. Open the Icechunk pyramid store from R2 and read dataset metadata
2. Build a `pystac.Item` template
3. Populate spatial / temporal extents with `xstac`
4. Add Icechunk asset with `zarr`, `virtual-assets`, and `storage` extension fields
5. Validate and save to `ndvi_cdr_pyramid_stac_item.json`
6. Round-trip: reconstruct the store from the saved item JSON

In [1]:
import json
import datetime
import os
import warnings

from dotenv import load_dotenv
import xarray as xr
import icechunk
import pystac
import xstac

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)

## 1. Open the Icechunk pyramid store from R2

Credentials are loaded from `~/dotenv/r2-osc-pub.env`.
We open the `main` branch and pin the **snapshot ID** so the STAC item
always refers to a reproducible version of the store.

In [2]:
load_dotenv(f'{os.environ["HOME"]}/dotenv/r2-osc-pub.env')

r2_bucket      = "osc-pub"
r2_prefix_pyr  = "ndvi-cdr-pyramid"
r2_prefix_src  = "ndvi-cdr-icechunk"
r2_endpoint    = os.environ["ENDPOINT_URL"]

# Level 0 of the pyramid has virtual refs → ndvi-cdr-icechunk, so we need
# a VirtualChunkContainer to resolve them when reading from the pyramid store.
pyr_storage = icechunk.s3_storage(
    bucket=r2_bucket,
    prefix=r2_prefix_pyr,
    from_env=True,
    endpoint_url=r2_endpoint,
    region="auto",
    force_path_style=False,
)

pyr_config = icechunk.RepositoryConfig.default()
pyr_config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f"s3://{r2_bucket}/{r2_prefix_src}/",
        store=icechunk.s3_store(
            endpoint_url=r2_endpoint,
            region="auto",
            force_path_style=False,
        ),
    ),
)

repo = icechunk.Repository.open(pyr_storage, pyr_config)
session = repo.readonly_session(branch="main")
snapshot_id = session.snapshot_id
print(f"snapshot_id: {snapshot_id}")

# Open level "0" for spatial/temporal metadata (root group has no coords)
ds = xr.open_zarr(session.store, group="0", consolidated=False, chunks=None)
ds

snapshot_id: PZAM8857J2XFJ4ZGX7NG


<xarray.Dataset> Size: 1GB
Dimensions:      (time: 5, latitude: 3600, longitude: 7200)
Coordinates:
  * time         (time) datetime64[ns] 40B 2000-01-01 2000-01-02 ... 2000-01-05
  * latitude     (latitude) float32 14kB 89.97 89.93 89.88 ... -89.93 -89.97
  * longitude    (longitude) float32 29kB -180.0 -179.9 -179.9 ... 179.9 180.0
    spatial_ref  int64 8B ...
Data variables:
    NDVI         (time, latitude, longitude) float64 1GB ...
Attributes: (12/32)
    institution:                            NASA/GSFC/SED/ESD/HBSL/TIS/MODIS-...
    Conventions:                            CF-1.6, ACDD-1.3
    standard_name_vocabulary:               CF Standard Name Table (v25, 05 J...
    naming_authority:                       gov.noaa.ncei
    license:                                See the Use Agreement for this CD...
    cdm_data_type:                          Grid
    ...                                     ...
    InputDataType:                          GAC
    ESDT:                                   AVH13C1
    RangeBeginningTime:                     00:00:00.0000
    RangeEndingTime:                        23:59:59.9999
    PercentValidClearDaytimeWater:          0.00
    PercentValidDaytimeWaterInCloudShadow:  0.00

## 2. Build the STAC Item template

In [3]:
item_id = f"noaa-cdr-ndvi-pyramid@{snapshot_id}"

start_dt = datetime.datetime.fromisoformat(str(ds.time.min().values))
end_dt   = datetime.datetime.fromisoformat(str(ds.time.max().values))

bbox = [-180.0, -90.0, 180.0, 90.0]
geometry = {
    "type": "Polygon",
    "coordinates": [[
        [-180.0, -90.0], [180.0, -90.0],
        [180.0,  90.0],  [-180.0,  90.0],
        [-180.0, -90.0],
    ]]
}

template = pystac.Item(
    id=item_id,
    geometry=geometry,
    bbox=bbox,
    datetime=None,
    properties={
        "start_datetime": start_dt.isoformat() + "Z",
        "end_datetime":   end_dt.isoformat()   + "Z",
        "title": "NOAA CDR NDVI — GeoZarr Multiscales Pyramid",
        "description": (
            "GeoZarr multiscales pyramid of the NOAA Climate Data Record (CDR) of AVHRR "
            "NDVI v5, stored as an Icechunk repository on Cloudflare R2. "
            "Levels 1–3 are fully materialised on R2; level 0 (full resolution) uses "
            "virtual chunk references pointing to the source Icechunk store "
            "`osc-pub/ndvi-cdr-icechunk`."
        ),
    },
    stac_extensions=[
        "https://stac-extensions.github.io/zarr/v1.1.0/schema.json",
        "https://stac-extensions.github.io/datacube/v2.2.0/schema.json",
        "https://stac-extensions.github.io/virtual-assets/v1.0.0/schema.json",
        "https://stac-extensions.github.io/version/v1.2.0/schema.json",
    ],
)
template

<Item id=noaa-cdr-ndvi-pyramid@PZAM8857J2XFJ4ZGX7NG>

## 3. Populate spatial / temporal extents with xstac

In [4]:
item = xstac.xarray_to_stac(
    ds,
    template,
    temporal_dimension="time",
    x_dimension="longitude",
    y_dimension="latitude",
    reference_system=4326,
    validate=False,
)

print("bbox:", item.bbox)
print("geometry type:", item.geometry["type"])
item

bbox: [-180.0, -90.0, 180.0, 90.0]
geometry type: Polygon


<Item id=noaa-cdr-ndvi-pyramid@PZAM8857J2XFJ4ZGX7NG>

## 4. Add the Icechunk pyramid asset

In [5]:
storage_schemes = {
    "r2-osc-pub": {
        "type": "aws-s3",
        "platform": r2_endpoint,
        "bucket": "osc-pub",
        "region": "not-used",
        "endpoint_url": r2_endpoint,
    },
}

item.extra_fields["storage:schemes"] = storage_schemes

In [6]:
out_path        = "ndvi_cdr_pyramid_stac_item.json"
pyr_asset_key   = f"ndvi-cdr-pyramid@{snapshot_id}"
src_asset_key   = "ndvi-cdr-icechunk"

# Primary asset: the pyramid store
item.add_asset(
    pyr_asset_key,
    pystac.Asset(
        href=f"s3://{r2_bucket}/{r2_prefix_pyr}/",
        title="NOAA CDR NDVI — GeoZarr Multiscales Pyramid (R2)",
        media_type="application/vnd.zarr+icechunk",
        roles=["data", "references", "virtual", "latest-version"],
        extra_fields={
            "zarr:consolidated": False,
            "zarr:zarr_format": 3,
            "icechunk:snapshot_id": snapshot_id,
            "storage:refs": ["r2-osc-pub"],
            # level 0 virtual refs resolve via the source icechunk store
            "vrt:hrefs": [
                {
                    "key": src_asset_key,
                    "href": f"./{out_path}#/assets/{src_asset_key}",
                }
            ],
        },
    ),
)

# Secondary asset: the source virtual icechunk store (needed to resolve level-0 chunks)
item.add_asset(
    src_asset_key,
    pystac.Asset(
        href=f"s3://{r2_bucket}/{r2_prefix_src}/",
        title="NOAA CDR NDVI — Source Virtual Icechunk Store (R2)",
        media_type="application/vnd.zarr+icechunk",
        roles=["data", "references", "virtual"],
        extra_fields={
            "zarr:consolidated": False,
            "zarr:zarr_format": 3,
            "storage:refs": ["r2-osc-pub"],
        },
    ),
)

item.assets

{'ndvi-cdr-pyramid@PZAM8857J2XFJ4ZGX7NG': <Asset href=s3://osc-pub/ndvi-cdr-pyramid/>,
 'ndvi-cdr-icechunk': <Asset href=s3://osc-pub/ndvi-cdr-icechunk/>}

## 5. Validate and save

In [7]:
item.validate()

with open(out_path, "w") as f:
    json.dump(item.to_dict(), f, indent=2)

print(f"Saved → {out_path}")
print(json.dumps(item.to_dict(), indent=2))

Saved → ndvi_cdr_pyramid_stac_item.json
{
  "type": "Feature",
  "stac_version": "1.1.0",
  "stac_extensions": [
    "https://stac-extensions.github.io/zarr/v1.1.0/schema.json",
    "https://stac-extensions.github.io/datacube/v2.2.0/schema.json",
    "https://stac-extensions.github.io/virtual-assets/v1.0.0/schema.json",
    "https://stac-extensions.github.io/version/v1.2.0/schema.json"
  ],
  "id": "noaa-cdr-ndvi-pyramid@PZAM8857J2XFJ4ZGX7NG",
  "geometry": {
    "type": "Polygon",
    "coordinates": [
      [
        [
          -180.0,
          -90.0
        ],
        [
          180.0,
          -90.0
        ],
        [
          180.0,
          90.0
        ],
        [
          -180.0,
          90.0
        ],
        [
          -180.0,
          -90.0
        ]
      ]
    ]
  },
  "bbox": [
    -180.0,
    -90.0,
    180.0,
    90.0
  ],
  "properties": {
    "start_datetime": "2000-01-01T00:00:00Z",
    "end_datetime": "2000-01-05T00:00:00Z",
    "title": "NOAA CDR NDVI

## 5b. Upload STAC item JSON to R2

Make the item discoverable at a public HTTP URL so the web app can fetch it.

In [8]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=r2_endpoint,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name="auto",
)

s3_key = out_path  # uploads as ndvi_cdr_pyramid_stac_item.json at bucket root

s3.upload_file(
    out_path,
    r2_bucket,
    s3_key,
    ExtraArgs={"ContentType": "application/json"},
)

public_stac_url = f"{r2_endpoint}/{r2_bucket}/{s3_key}"
print(f"STAC item uploaded → {public_stac_url}")

STAC item uploaded → https://9cbdcb4884f86a6779032ae561e474a5.r2.cloudflarestorage.com/osc-pub/ndvi_cdr_pyramid_stac_item.json


## 6. Round-trip: open dataset from saved STAC item JSON

Reconstruct the public HTTP URL from `storage:schemes` + asset `href`,
then open via `IcechunkStore` (same path the JS app uses).

In [9]:
loaded_item = pystac.Item.from_file(out_path)

# Find the pyramid asset (has icechunk:snapshot_id)
pyr_asset = next(
    a for a in loaded_item.assets.values()
    if "icechunk:snapshot_id" in a.extra_fields
)
snap_id  = pyr_asset.extra_fields["icechunk:snapshot_id"]
ref      = pyr_asset.extra_fields["storage:refs"][0]
scheme   = loaded_item.extra_fields["storage:schemes"][ref]

bucket   = scheme["bucket"]
endpoint = scheme["endpoint_url"]
prefix   = pyr_asset.href.split(f"{bucket}/")[1].rstrip("/")

public_url = f"{endpoint}/{bucket}/{prefix}"
print(f"Public HTTP URL: {public_url}")
print(f"snapshot:        {snap_id}")

# Source store prefix (for virtual chunk container at level 0)
src_asset  = loaded_item.assets["ndvi-cdr-icechunk"]
src_prefix = src_asset.href.split(f"{bucket}/")[1].rstrip("/")

storage2 = icechunk.s3_storage(
    bucket=bucket,
    prefix=prefix,
    from_env=True,
    endpoint_url=endpoint,
    region="auto",
    force_path_style=False,
)

config2 = icechunk.RepositoryConfig.default()
config2.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f"s3://{bucket}/{src_prefix}/",
        store=icechunk.s3_store(
            endpoint_url=endpoint,
            region="auto",
            force_path_style=False,
        ),
    ),
)

repo2    = icechunk.Repository.open(storage2, config2)
session2 = repo2.readonly_session(snapshot_id=snap_id)
ds2      = xr.open_zarr(session2.store, consolidated=False, chunks=None)
ds2

Public HTTP URL: https://9cbdcb4884f86a6779032ae561e474a5.r2.cloudflarestorage.com/osc-pub/ndvi-cdr-pyramid
snapshot:        PZAM8857J2XFJ4ZGX7NG


<xarray.Dataset> Size: 0B
Dimensions:  ()
Data variables:
    *empty*
Attributes:
    zarr_conventions:    [{'schema_url': 'https://raw.githubusercontent.com/z...
    multiscales:         {'layout': [{'asset': '0', 'transform': {'scale': [1...
    proj:code:           EPSG:4326
    spatial:dimensions:  ['latitude', 'longitude']
    spatial:transform:   [0.0500030517578125, 0.0, -180.00000762939453, 0.0, ...
    spatial:bbox:        [-180.00000762939453, -89.98352432250977, 180.021965...
    spatial:shape:       [3600, 7200]